# Notebook 5 — Peer Comparison Engine
For any given company, find its top 5 financial peers using cosine similarity on the feature matrix. Exports `data/peer_mapping.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})

DATA = Path('../data/clean')
pl = pd.read_csv(DATA / 'profitandloss.csv')
bs = pd.read_csv(DATA / 'balancesheet.csv')
cf = pd.read_csv(DATA / 'cashflow.csv')
co = pd.read_csv(DATA / 'companies.csv')
an = pd.read_csv(DATA / 'analysis.csv')

pl = pl[pl['is_ttm'] == False]
bs = bs[bs['is_ttm'] == False]
cf = cf[cf['is_ttm'] == False]
print('Data loaded.')

## Step 1 — Build feature matrix (same as Notebook 4)

In [ ]:
avg_pl = pl.groupby('company_id').agg(
    avg_opm=('opm_percentage','mean'), avg_npm=('net_profit_margin_pct','mean'),
    avg_div=('dividend_payout','mean'), avg_ic=('interest_coverage','mean')
).reset_index()
avg_bs = bs.groupby('company_id').agg(
    avg_de=('debt_to_equity','mean'), avg_eqr=('equity_ratio','mean')
).reset_index()
avg_cf = cf.groupby('company_id').agg(
    avg_ocf=('operating_activity','mean'), avg_fcf=('free_cash_flow','mean')
).reset_index()
an3 = an[an['period']=='3Y'][['company_id','compounded_sales_growth_pct']].rename(
    columns={'compounded_sales_growth_pct':'growth_3y'})
roe_df = co[['symbol','roe_percentage']].rename(columns={'symbol':'company_id'})

feats = avg_pl.merge(avg_bs,on='company_id',how='outer')
feats = feats.merge(avg_cf,on='company_id',how='outer')
feats = feats.merge(an3,on='company_id',how='left')
feats = feats.merge(roe_df,on='company_id',how='left')
feats = feats.set_index('company_id').fillna(feats.median(numeric_only=True))

scaler = StandardScaler()
X = scaler.fit_transform(feats)
print(f'Feature matrix: {X.shape}')

## Step 2 — Compute cosine similarity matrix

In [ ]:
sim_matrix = cosine_similarity(X)
sim_df = pd.DataFrame(sim_matrix, index=feats.index, columns=feats.index)
print(f'Similarity matrix: {sim_df.shape}')
print('\nSample: TCS similarity to others (top 10):')
if 'TCS' in sim_df.index:
    print(sim_df['TCS'].sort_values(ascending=False).head(10))

## Step 3 — Find top 5 peers for every company

In [ ]:
peer_records = []
for sym in feats.index:
    peers = sim_df[sym].drop(sym).nlargest(5)
    for rank, (peer_sym, sim_score) in enumerate(peers.items(), 1):
        peer_records.append({'symbol': sym, 'peer_rank': rank,
                             'peer_symbol': peer_sym, 'similarity_score': round(sim_score, 4)})

peer_mapping = pd.DataFrame(peer_records)
print(f'Peer mappings generated: {len(peer_mapping)}')
print(peer_mapping[peer_mapping['symbol'] == 'TCS'].to_string(index=False))

## Step 4 — Validation: TCS peers should be IT companies

In [ ]:
companies_map = co.set_index('symbol')['sector'].to_dict()
for test_sym in ['TCS', 'HDFCBANK', 'RELIANCE']:
    if test_sym not in peer_mapping['symbol'].values:
        print(f'{test_sym} not in data')
        continue
    peers = peer_mapping[peer_mapping['symbol'] == test_sym]
    peer_syms = peers['peer_symbol'].tolist()
    peer_sectors = [companies_map.get(p, 'Unknown') for p in peer_syms]
    print(f'\n{test_sym} ({companies_map.get(test_sym,"?")}) — top 5 peers:')
    for p, s, sc in zip(peer_syms, peer_sectors, peers['similarity_score'].tolist()):
        print(f'  {p:<15} ({s:<15}) similarity={sc:.4f}')

## Step 5 — Similarity heatmap for top 20 companies

In [ ]:
top20 = feats.index[:20].tolist()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_df.loc[top20, top20], cmap='RdYlGn', vmin=0, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Cosine similarity'})
ax.set_title('Financial similarity heatmap — top 20 companies')
plt.tight_layout()
plt.show()

## Step 6 — Export peer_mapping.csv

In [ ]:
peer_mapping.to_csv('../data/peer_mapping.csv', index=False)
print(f'Exported data/peer_mapping.csv — {len(peer_mapping)} rows')
print(peer_mapping.head(10).to_string(index=False))